# 04 · Auditoría & Gobernanza

**Configuración base:**
- Catálogo: `main`
- Schemas: `loterias_raw`, `loterias_bronze`, `loterias_silver`, `loterias_features`
- Volume crudos: `/Volumes/main/loterias_raw/raw_apuestas/apuestas_partitioned/`

In [0]:
display(spark.sql("DESCRIBE HISTORY main.loterias_bronze.apuestas_bronze"))
display(spark.sql("DESCRIBE HISTORY main.loterias_silver.apuestas_silver"))
display(spark.sql("DESCRIBE HISTORY main.loterias_features.features_apuestas"))

In [0]:
from pyspark.sql import functions as F
df = spark.table("main.loterias_silver.apuestas_silver")
checks = {
    "monto>0": df.filter(F.col("monto") <= 0).count()==0,
    "monto<10000": df.filter(F.col("monto") >= 10000).count()==0,
    "tx_id_unicos": df.select("tx_id").distinct().count()==df.count(),
    "user_id_no_nulos": df.filter(F.col("user_id").isNull()).count()==0,
}
print(checks)
assert all(checks.values()), "Falla en controles de calidad"